### Create DataFrame

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("FlipkartPipeline").getOrCreate()

### LOAD DATASET

In [0]:
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]
df=spark.createDataFrame(data,columns)
df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  NULL|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|   

In [0]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- quantity: long (nullable = true)



### BRONZE LAYER

In [0]:
path_bronze='/Volumes/bronze_flipkart/default/silver_deltatable'
df.write.format('delta').mode('overwrite').option('overwriteSchema',"true").save(path_bronze)

In [0]:
df_bronze = spark.read.format("delta").load(path_bronze)

### SILVER LAYER

### Handling NULL Values 

In [0]:
from pyspark.sql.functions import *
df_clean=df.withColumn('amount',when(col('amount').isNull(),0).otherwise(col("amount")))

In [0]:
df_clean.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


### Fix DataType

In [0]:
df_clean=df_clean.withColumn('amount',col('amount').cast('int'))\
    .withColumn('date',col('date').cast('date'))

In [0]:
df_clean.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: date (nullable = true)
 |-- amount: integer (nullable = true)
 |-- quantity: long (nullable = true)



### Removing Negative Values

In [0]:
df_clean=df_clean.filter(col('amount')>0)

In [0]:
df_clean.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+---

### Removing Duplicates

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
window=Window.partitionBy('order_id').orderBy(col('date').desc())
df_clean=df_clean.withColumn('row_num',row_number().over(window))\
    .filter(col('row_num')==1)\
    .drop('row_num')

In [0]:
df_clean.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+--------+-----------+----------+-----------+---------+----------+------+--------+



### Handling NULL City

In [0]:
df_clean=df_clean.fillna({'city':'Unknown'})

In [0]:
df_clean.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|  Unknown|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+--------+-----------+----------+-----------+---------+----------+------+--------+



### Save Silver Data

In [0]:
path_silver="/Volumes/bronze_flipkart/default/silver2_deltatable"
df_clean.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .save(path_silver)

### GOLD LAYER

### Total Sales Of Product

In [0]:
from pyspark.sql.functions import sum
df_silver = spark.read.format("delta").load(path_silver)
sales = df_silver.groupBy("product") \
                 .agg(sum("amount").alias("total_sales"))

In [0]:
sales.display()

product,total_sales
Tablet,22000
Mobile,33000
Watch,8000
Laptop,105000
Headphones,3000


### Sales By Category

In [0]:
sales_category=df_silver.groupBy("category").agg(sum("amount").alias("total_sales"))
sales_category.display()

category,total_sales
Electronics,160000
Accessories,11000


### Total Sales By City

In [0]:
sales_city=df_silver.groupBy('city').agg(sum('amount').alias('total_sales'))

In [0]:
sales_city.display()

city,total_sales
Delhi,55000
Chennai,33000
Unknown,3000
Hyderabad,72000
Mumbai,8000


### No.of Order Per Customer

In [0]:
order_customer=df_silver.groupBy('customer_id').count().withColumnRenamed('count','total_orders')

In [0]:
order_customer.display()

customer_id,total_orders
C005,1
C004,1
C003,1
C001,2
C002,1
C007,1


### Total Spending for Customers

In [0]:
spend_customers=df_silver.groupBy('customer_id').agg(sum('amount').alias('total_spend'))
spend_customers.display()

customer_id,total_spend
C005,3000
C004,8000
C003,55000
C001,72000
C002,18000
C007,15000


### Top Customers (Highest Spending)

In [0]:
top_customers=spend_customers.orderBy(col('total_spend').desc())
top_customers.display()

customer_id,total_spend
C001,72000
C003,55000
C002,18000
C007,15000
C004,8000
C005,3000


### Top Selling Product

In [0]:
top_product=sales.orderBy(col('total_sales').desc()).limit(1)
top_product.display()

product,total_sales
Laptop,105000


### Save Gold Layer

In [0]:
path_gold = "/Volumes/bronze_flipkart/default/gold_deltatable"
sales.write.format("delta").mode("overwrite").save(path_gold + "/sales_product")
sales_category.write.format("delta").mode("overwrite").save(path_gold + "/sales_category")
sales_city.write.format("delta").mode("overwrite").save(path_gold + "/sales_city")